# Notebook 5: Regression Validation 

This notebook validates the resemblance list by re-deriving it from a regression on the continuous milieu_anteil, instead of the binary >50% classification used in NB1–NB3. 

The pipeline is otherwise identical — same features, same GroupKFold-by-district out-of-fold scheme, same leakage-safe preprocessing — so any difference in the resulting shortlist is attributable to the target form (continuous share vs binary majority), not to a change in method. 

If the two approaches rank largely the same undesignated areas, the resemblance list is robust to that modelling choice; where they disagree tells us which part of the ranking is genuinely sensitive.

In [1]:
#load libraries
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from sklearn.model_selection import GroupKFold
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor
from Dataprep import preprocess_fit_transform


In [2]:
#paths (same relative layout as NB3) + parameters

FEATURES_PATH = "../../data/final_datasets/dataset_0.csv"
TARGET_PATH   = "../../data/Module 2/target_milieuschutz.csv"
CLF_OOF_PATH  = "../../models/M_oof_xgboost.csv"      

N_WATCHLIST = 25
N_SPLITS    = 5          

In [3]:
#load features + target, zero-pad plr_id, merge continuous milieu_anteil
feat = pd.read_csv(FEATURES_PATH, dtype={"plr_id": str})
tgt  = pd.read_csv(TARGET_PATH,   dtype={"plr_id": str})
feat["plr_id"] = feat["plr_id"].astype(str).str.zfill(8)
tgt["plr_id"]  = tgt["plr_id"].astype(str).str.zfill(8)

data = feat.merge(tgt[["plr_id", "milieu_anteil"]], on="plr_id", how="left")
assert data["milieu_anteil"].notna().all(), "some PLR has no milieu_anteil after merge"

In [4]:
#build X / y / groups (mirrors NB1/NB2 column definitions)
ID_COLS      = ["plr_id", "plr_name", "bez"]
GROUP_COL    = "bez"
LABEL        = "milieu_majoritaet"     
LEAKAGE_COLS = ["milieu_anteil"]       

feature_cols = [
    c for c in data.columns
    if c not in ID_COLS + [LABEL] + LEAKAGE_COLS
]
assert not any("milieu" in c for c in feature_cols), \
    "column derived from target is in feature-set — Leakage!"
assert GROUP_COL not in feature_cols, "bez must be group only, not a feature"

X = data[feature_cols].copy()
y = data["milieu_anteil"].to_numpy()     
groups = data[GROUP_COL].to_numpy()

print(f"{len(data)} PLR | {len(feature_cols)} features | "
      f"target range {y.min():.3f}-{y.max():.3f} | "
      f"share at 0: {(y == 0).mean():.2%}")

527 PLR | 42 features | target range 0.000-1.000 | share at 0: 59.01%


In [5]:
#OOF regression loop (unrolled, GroupKFold by district)
#Preprocessing via preprocess_fit_transform, identical args to NB2: KNN impute only, no log, no scale.
#Regressor mirrors NB2's minimal classifier (defaults + housekeeping); only the objective is swapped.
oof_pred = np.full(len(data), np.nan)
gkf = GroupKFold(n_splits=N_SPLITS)

for fold, (train_idx, val_idx) in enumerate(gkf.split(X, y, groups), start=1):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train = y[train_idx]

    #fold-wise preprocessing (leakage-safe): fit imputer on train, apply to both
    X_train_prep, X_val_prep = preprocess_fit_transform(
        X_train, X_val, feature_cols, log_cols=[], scale=False
    )

    model = XGBRegressor(
        objective="reg:logistic",   # target in [0,1]; regression analogue of the classifier
        random_state=42,
        n_jobs=-1,
    )
    model.fit(X_train_prep, y_train)
    oof_pred[val_idx] = model.predict(X_val_prep)

    #fold diagnosis: rank agreement on the held-out fold + which districts
    fold_rho, _ = spearmanr(oof_pred[val_idx], y[val_idx])
    print(f"Fold {fold}: n_val={len(val_idx):3d} | "
          f"Spearman={fold_rho:.3f} | districts={np.unique(groups[val_idx])}")

data["oof_pred"] = oof_pred
data["residual"] = data["oof_pred"] - data["milieu_anteil"]
assert data["oof_pred"].notna().all(), "some PLR never got an OOF prediction"

Fold 1: n_val= 96 | Spearman=0.772 | districts=['03 - Pankow' '10 - Marzahn-Hellersdorf']
Fold 2: n_val= 92 | Spearman=0.519 | districts=['04 - Charlottenburg-Wilmersdorf' '11 - Lichtenberg']
Fold 3: n_val=125 | Spearman=0.749 | districts=['01 - Mitte' '09 - Treptow-Köpenick' '12 - Reinickendorf']
Fold 4: n_val= 91 | Spearman=0.726 | districts=['05 - Spandau' '07 - Tempelhof-Schöneberg']
Fold 5: n_val=123 | Spearman=0.776 | districts=['02 - Friedrichshain-Kreuzberg' '06 - Steglitz-Zehlendorf'
 '08 - Neukölln']


In [6]:
#regression quality (rank-based, since this is a filter not an estimator)
rho, _ = spearmanr(data["oof_pred"], data["milieu_anteil"])
mae = np.abs(data["residual"]).mean()
print(f"pooled Spearman(oof_pred, milieu_anteil) = {rho:.3f} | MAE = {mae:.3f}")

pooled Spearman(oof_pred, milieu_anteil) = 0.705 | MAE = 0.138


In [7]:
#candidate set = undesignated only (same gate as classification: anteil <= 0.5)
#rank by positive residual = "more designation-like than its protection"
cand = data[data["milieu_anteil"] <= 0.50].copy()
cand["rank_resid"] = cand["residual"].rank(ascending=False, method="min").astype(int)
cand["rank_pred"]  = cand["oof_pred"].rank(ascending=False, method="min").astype(int)

resid_top = set(cand.nsmallest(N_WATCHLIST, "rank_resid")["plr_id"])
pred_top  = set(cand.nsmallest(N_WATCHLIST, "rank_pred")["plr_id"])

In [8]:
#compare against the classification RESEMBLANCE watchlist (oof_prob arm only),
#reconstructed from the classification OOF so it doesn't depend on how M2_watchlist1
#was assembled. Undesignated PLR, top-25 by oof_prob = NB3's oof_prob arm.
#All three sets are now single top-25s -> overlaps are symmetric (X/25 both ways).
clf_oof = pd.read_csv(CLF_OOF_PATH, dtype={"plr_id": str})
clf_oof["plr_id"] = clf_oof["plr_id"].astype(str).str.zfill(8)

clf_resemblance = set(
    clf_oof[clf_oof["y_true"] == 0]
    .nlargest(N_WATCHLIST, "oof_prob")["plr_id"]
)
assert len(clf_resemblance) == N_WATCHLIST, "classification resemblance set is not 25"

#how correlated are the two model outputs at the PLR level?
merged = data.merge(clf_oof[["plr_id", "oof_prob"]], on="plr_id", how="inner")
rho_models, _ = spearmanr(merged["oof_pred"], merged["oof_prob"])


def overlap(a, b, label_a, label_b):
    inter = a & b
    print(f"{label_a}  vs  {label_b}: {len(inter)}/{len(a)} overlap "
          f"(only {label_a}: {sorted(a - b)}) "
          f"(only {label_b}: {sorted(b - a)})")


print(f"Spearman(regression oof_pred, classification oof_prob) = {rho_models:.3f}\n")
overlap(pred_top,  clf_resemblance, "reg oof_pred top-25", "clf oof_prob top-25")
overlap(resid_top, clf_resemblance, "reg residual top-25", "clf oof_prob top-25")
overlap(resid_top, pred_top,        "reg residual top-25", "reg oof_pred top-25")

Spearman(regression oof_pred, classification oof_prob) = 0.905

reg oof_pred top-25  vs  clf oof_prob top-25: 20/25 overlap (only reg oof_pred top-25: ['01100415', '03601452', '07200413', '09100305', '12100103']) (only clf oof_prob top-25: ['01200628', '04300416', '04501147', '04501150', '12100102'])
reg residual top-25  vs  clf oof_prob top-25: 17/25 overlap (only reg residual top-25: ['01100206', '01100415', '01200517', '01200629', '05300841', '09100305', '09200716', '12100103']) (only clf oof_prob top-25: ['01200628', '03601142', '04501150', '07100206', '07200414', '08100102', '09200511', '12100102'])
reg residual top-25  vs  reg oof_pred top-25: 18/25 overlap (only reg residual top-25: ['01100206', '01200517', '01200629', '04300416', '04501147', '05300841', '09200716']) (only reg oof_pred top-25: ['03601142', '03601452', '07100206', '07200413', '07200414', '08100102', '09200511'])


In [9]:
data.loc[data.plr_id.isin(['12100102','12100103']), ['plr_id','plr_name']]

,plr_id,plr_name
494,12100102,Hausotterplatz
495,12100103,Breitkopfbecken


In [10]:
#Diagnostic: are the swap PLR near-cutoff ties or real disagreements?
#Show oof_pred (reg) + oof_prob (clf) side by side with both ranks, for the
#union of the two top-25s, so the rank-25 boundary is visible.

#reconstruct classification resemblance ranking with ranks (undesignated only)
clf_res = (
    clf_oof[clf_oof["y_true"] == 0][["plr_id", "oof_prob"]]
    .copy()
)
clf_res["rank_prob"] = clf_res["oof_prob"].rank(ascending=False, method="min").astype(int)

#regression raw-resemblance ranking with ranks (same undesignated gate)
reg_res = cand[["plr_id", "plr_name", "bez", "oof_pred", "milieu_anteil"]].copy()
reg_res["rank_pred"] = reg_res["oof_pred"].rank(ascending=False, method="min").astype(int)

#join the two views
cmp = reg_res.merge(clf_res, on="plr_id", how="inner")

#the PLR that differ between reg oof_pred top-25 and clf oof_prob top-25
swap = sorted(pred_top ^ clf_resemblance)          # symmetric difference
view = (cmp[cmp["plr_id"].isin(swap)]
        .sort_values("rank_pred")
        [["plr_id", "plr_name", "bez", "oof_pred", "rank_pred", "oof_prob", "rank_prob", "milieu_anteil"]])

print(f"{len(swap)} swap PLR (in exactly one of the two raw-resemblance top-25s):\n")
print(view.to_string(index=False))

#how tight is the rank-25 boundary in each model?
p_in  = cand.nlargest(25, "oof_pred")["oof_pred"].iloc[-1]
p_out = cand.nlargest(26, "oof_pred")["oof_pred"].iloc[-1]
q_in  = clf_res.nsmallest(25, "rank_prob")["oof_prob"].iloc[-1]
q_out = clf_res.nsmallest(26, "rank_prob")["oof_prob"].iloc[-1]
print(f"\nreg oof_pred cutoff: #25={p_in:.3f} | #26={p_out:.3f} | gap={p_in-p_out:.4f}")
print(f"clf oof_prob cutoff: #25={q_in:.3f} | #26={q_out:.3f} | gap={q_in-q_out:.4f}")

10 swap PLR (in exactly one of the two raw-resemblance top-25s):

  plr_id           plr_name                             bez  oof_pred  rank_pred  oof_prob  rank_prob  milieu_anteil
03601452           Velodrom                     03 - Pankow  0.690303         10  0.008326        136       0.361355
07200413    Cheruskerstraße       07 - Tempelhof-Schöneberg  0.515034         18  0.100890         69       0.295770
12100103    Breitkopfbecken              12 - Reinickendorf  0.482257         19  0.712730         29       0.000000
09100305 Baumschulenweg Ost           09 - Treptow-Köpenick  0.446441         24  0.447517         46       0.000000
01100415    Invalidenstraße                      01 - Mitte  0.436678         25  0.693884         32       0.000000
04300416  Amtsgerichtsplatz 04 - Charlottenburg-Wilmersdorf  0.415842         28  0.862824         22       0.018465
04501147  Leon-Jessel-Platz 04 - Charlottenburg-Wilmersdorf  0.408259         29  0.984600          7       0.00000

In [12]:
data.loc[data.plr_id.isin(['01200628','04300416','04501147','04501150','12100102']),
         ['plr_id','plr_name','milieu_anteil','oof_pred']]

,plr_id,plr_name,milieu_anteil,oof_pred
27,01200628,Lüneburger Straße,0.000000,0.351868
155,04300416,Amtsgerichtsplatz,0.018465,0.415842
185,04501147,Leon-Jessel-Platz,0.000000,0.408259
188,04501150,Wilhelmsaue,0.000000,0.152387
494,12100102,Hausotterplatz,0.000000,0.311000


In [11]:
#names for the residual watchlist, for eyeballing
print(
    cand.nsmallest(N_WATCHLIST, "rank_resid")
    [["plr_id", "plr_name", "bez", "oof_pred", "milieu_anteil", "residual"]]
    .to_string(index=False)
)

  plr_id              plr_name                             bez  oof_pred  milieu_anteil  residual
01300834         Brunnenstraße                      01 - Mitte  0.749736   1.430044e-04  0.749593
01100416           Arkonaplatz                      01 - Mitte  0.689894   9.594333e-04  0.688935
01401049           Schulstraße                      01 - Mitte  0.889269   2.382715e-01  0.650997
07300619          Grazer Platz       07 - Tempelhof-Schöneberg  0.805050   1.870982e-01  0.617952
01300836 Humboldthain Nordwest                      01 - Mitte  0.711323   1.296414e-01  0.581682
01400938      Kameruner Straße                      01 - Mitte  0.803364   2.381216e-01  0.565243
04300414          Schloßstraße 04 - Charlottenburg-Wilmersdorf  0.594934   3.928747e-02  0.555647
01300731         Koloniestraße                      01 - Mitte  0.733899   2.211197e-01  0.512779
09100101           Alt-Treptow           09 - Treptow-Köpenick  0.681266   1.842958e-01  0.496970
12100103       Breit

### Results and Decisions 

Comparing the regression's raw resemblance ranking (oof_pred) against the classification's resemblance ranking (oof_prob), both on undesignated areas, the two top-25 lists overlap by 20 of 25, and the underlying scores correlate at Spearman 0.905 (a figure flattered by the designated areas both models trivially agree on, so the undesignated-tail overlap is the more honest measure). 

Switching the target from the binary majority label to the continuous share therefore barely moves the shortlist: the target form is close to immaterial. The sensitivity that does exist lies not in the target but in whether the ranking is discounted by existing protection — applying that discount reshuffles about 7 of the top 25, the same magnitude seen in NB3's oof_prob-vs-urgency comparison, confirming it is the discounting choice, not the model, that governs those cases. 

The classification model stays primary because its scale_pos_weight ranks undesignated, zero-protection areas highly, whereas the regression partly absorbs the protection signal into the target itself and ranks those same areas lower. 

The areas the classification ranks high but the regression drops are all zero- or near-zero-protection (Lüneburger Straße, Amtsgerichtsplatz, Leon-Jessel-Platz, Wilhelmsaue, Hausotterplatz, all milieu_anteil ≤ 0.02), including Leon-Jessel-Platz, which the local parliament in Charlottenburg-Wilmersdorf designated during the period of the project. The regression thus corroborates the watchlist as equivalent under a different target — the point of a robustness check — without displacing the classifier.